# Four-Format Data Recovery and Validation — Group 5

**Dataset:** Gapminder historical teaching snapshot (1952–2007)  
**Purpose:** Demonstrate source recovery, validation, agreement across formats, NumPy analysis, and auditable publication.

Run this notebook **from the project root**, alongside `main.py` and `config.json`. Install dependencies from `requirements.txt` and run all cells before exporting to HTML. All figures below are computed from the actual generated outputs, not manually entered results.

## 1. Environment and project imports

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display

from src.pipeline import run_pipeline
from src.exporter import compare_frames, COLUMNS
from src.analysis import (
    asia_selected_year, asia_pop, country_history,
    population_matrix, calc_verification,
)

ROOT = Path.cwd()
assert (ROOT / "main.py").exists(), "Open the notebook from the project root."
with (ROOT / "config.json").open(encoding="utf-8") as handle:
    config = json.load(handle)
with (ROOT / "manifest.json").open(encoding="utf-8") as handle:
    manifest = json.load(handle)

# Resolve config paths in case they are relative to the project root.
config = dict(config)
for key in ("data_dir", "output_dir", "manifest"):
    if key in config:
        path = Path(config[key])
        if not path.is_absolute():
            config[key] = str((ROOT / path).resolve())

output = Path(config["output_dir"])
reporting_year = config.get("reporting_year", 2007)
print("Reporting year:", reporting_year)
print("Output directory:", output)
print("Input formats:", [item["format"] for item in manifest["files"]])

## 2. Execute the existing recovery pipeline

This calls the same code as `py main.py --config config.json`: SHA-256 verification, mapping, quarantine, deduplication, cross-format comparison, analysis, four-format export/reload checks, and staged publication.

In [ ]:
run_pipeline(config)
print("Pipeline execution: PASS")

## 3. Recovery and reconciliation counts

Each source has the same underlying observations. The four cleaned sources are **compared**, not appended together.

In [ ]:
canonical = pd.read_csv(output / "canonical.csv")
quarantine = pd.read_csv(output / "quarantine.csv")
duplicates = pd.read_csv(output / "duplicate_log.csv")
validation = pd.read_csv(output / "validation.csv")
audit = pd.read_csv(output / "audit_log.csv")

expected = manifest["expected_per_file"]
recovery = pd.DataFrame({
    "measure": ["Raw rows per source", "Rejected per source", "Exact duplicates per source",
                "Clean rows per source", "Final canonical rows", "Quarantine rows across sources",
                "Duplicate rows across sources"],
    "expected": [expected["raw_rows"], expected["rejected_rows"],
                 expected["exact_duplicates_removed"], expected["canonical_rows"],
                 expected["canonical_rows"], 4*expected["rejected_rows"],
                 4*expected["exact_duplicates_removed"]],
    "actual": [int(expected["raw_rows"]), len(quarantine)//4, len(duplicates)//4,
               len(canonical), len(canonical), len(quarantine), len(duplicates)],
})
recovery["matches"] = recovery["expected"].eq(recovery["actual"])
display(recovery)
assert recovery["matches"].all(), "Recovery counts do not match the manifest."
print("Quarantine fields:", quarantine.columns.tolist())
print("Duplicate-log fields:", duplicates.columns.tolist())

**Interpretation.** Each input format is expected to contain 1,712 records. Six invalid records and two exact duplicates per format leave 1,704 canonical observations. The combined quarantine and duplicate logs contain 24 and 8 entries respectively, while the final canonical dataset still contains 1,704 observations.

## 4. Verify cross-format agreement and export round trips

`compare_frames` sorts by `(country, year)`, requires exact text and integer agreement, and compares floating-point columns with absolute tolerance `1e-8` and zero relative tolerance.

In [ ]:
reloaded = {
    "CSV": pd.read_csv(output / "canonical.csv"),
    "Excel": pd.read_excel(output / "canonical.xlsx", sheet_name="observations"),
    "JSON": pd.read_json(output / "canonical.json", orient="records"),
    "Parquet": pd.read_parquet(output / "canonical.parquet"),
}
reference = reloaded["CSV"]
checks = []
for name, frame in reloaded.items():
    compare_frames(reference[COLUMNS], frame[COLUMNS])
    checks.append({"export": name, "rows": len(frame), "round_trip_agreement": "PASS"})
display(pd.DataFrame(checks))

source_agreement = validation[
    validation["check"].astype(str).str.contains("CSV vs", case=False, regex=False)
]
print("Source agreement checks recorded by pipeline:")
display(source_agreement)
assert len(source_agreement) == 3, "Expected CSV vs Excel, JSON, and Parquet checks."
assert source_agreement["pass"].astype(str).str.lower().eq("true").all()

**Interpretation.** The pipeline records agreement of the cleaned Excel, JSON, and Parquet sources against the cleaned CSV source. Separately, all four published canonical exports are reloaded and compared with the CSV canonical table. Passing these checks supports consistency across file formats.

## 5. Canonical dataset integrity

In [ ]:
display(canonical.head())
print("Rows:", len(canonical))
print("Countries:", canonical["country"].nunique())
print("Duplicate country-year keys:", canonical.duplicated(["country", "year"]).sum())
print("Canonical columns:", canonical.columns.tolist())
assert canonical.columns.tolist() == COLUMNS
assert len(canonical) == 1704
assert canonical.duplicated(["country", "year"]).sum() == 0

## 6. Asia selected-year analysis and NumPy verification

In [ ]:
asia = asia_selected_year(canonical, {"reporting_year": reporting_year})
pandas_total = asia_pop(asia)
numpy_total = int(np.sum(asia["pop"].to_numpy(dtype=np.int64)))
print(f"Asia {reporting_year} rows: {len(asia)}")
print(f"Asia {reporting_year} population (pandas): {pandas_total:,}")
print(f"Asia {reporting_year} population (NumPy): {numpy_total:,}")
assert pandas_total == numpy_total
if reporting_year == 2007:
    assert len(asia) == manifest["controls"]["asia_2007_rows"]
    assert pandas_total == manifest["controls"]["asia_2007_population"]
display(asia[["country", "year", "lifeExp", "pop", "pop_millions", "lifeExp_rank"]].head(10))

**Interpretation.** The selected-year table filters Asian countries and ranks life expectancy from highest to lowest. Pandas and NumPy produce the same total population. The benchmark of **3,811,953,827** applies specifically to 2007; other configured years will have different totals.

## 7. Philippines history and population controls

In [ ]:
history = country_history(canonical)
verification = calc_verification(canonical)
print("Philippines historical rows:", len(history))
print("Asia 2007 population control:", verification["calculated_asia_pop"])
print("Philippines 2007 population control:", verification["calculated_ph_pop"])
assert len(history) == manifest["controls"]["philippines_rows"]
assert verification["asia_population_matches_benchmark"]
assert verification["ph_population_matches_benchmark"]
display(history[["country", "year", "lifeExp", "lifeExp_change", "pop", "pop_millions"]])

**Interpretation.** The Philippines history contains 12 five-year observations. The `lifeExp_change` column measures the change relative to the first observed year. The independent 2007 controls confirm the expected Philippines and Asia population totals.

## 8. NumPy population index matrix (broadcasting)

In [ ]:
index_df = population_matrix(canonical)
display(index_df)
index_values = index_df.drop(columns="country").to_numpy(dtype=float)
print("Numeric matrix shape:", index_values.shape)
print("Countries:", index_df["country"].tolist())
assert index_values.shape == (3, 6)
np.testing.assert_allclose(index_values[:, 0], 100.0, rtol=0, atol=1e-8)

# Independent NumPy broadcasting verification from raw population observations.
countries = ["Philippines", "Indonesia", "Thailand"]
years = [1982, 1987, 1992, 1997, 2002, 2007]
raw_matrix = (canonical[canonical["country"].isin(countries) & canonical["year"].isin(years)]
              .pivot(index="country", columns="year", values="pop")
              .reindex(index=countries, columns=years)
              .to_numpy(dtype=float))
independent_index = raw_matrix / raw_matrix[:, :1] * 100
np.testing.assert_allclose(index_values, independent_index, rtol=0, atol=1e-8)
mask = raw_matrix > 100_000_000
print("Observations above 100 million:", int(mask.sum()))
print("Mean of selected population observations:", float(raw_matrix[mask].mean()) if mask.any() else "N/A")
print("Broadcasting comparison: PASS")

**Interpretation.** The 3×6 numeric matrix represents three countries across six years (1982–2007). NumPy broadcasting divides each country's population series by its own 1982 baseline, then multiplies by 100. The baseline therefore equals 100 for each country. Boolean indexing selects observations above 100 million for an additional aggregate.

## 9. Validation, quarantine, duplicates, and audit evidence

In [ ]:
print("Validation checks:", len(validation))
passed = validation["pass"].astype(str).str.lower().eq("true")
print("Passing checks:", int(passed.sum()))
print("All validation checks pass:", bool(passed.all()))
assert passed.all(), "One or more validation checks failed."
display(validation)
print("Audit events:", len(audit))
display(audit)
print("Quarantined rows (sample):")
display(quarantine.head())
print("Duplicate rows (sample):")
display(duplicates.head())

## 10. Conclusion

The four-format pipeline verifies source integrity, normalizes the data to a common schema, quarantines invalid records, removes exact duplicates, and compares cleaned datasets before publication. The canonical outputs pass reload comparisons, and the analytical controls agree with independent NumPy calculations. The validation, quarantine, duplicate, and audit reports provide a reproducible record of the recovery process.

**Scope note:** These are historical teaching observations through 2007, not current population estimates. The notebook reflects the configuration and data used when its cells were executed.